<a href="https://colab.research.google.com/github/noname123920/diplom/blob/main/pinn_deepxde2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [45]:
!pip install deepxde
!pip install optuna

In [46]:
import os
os.environ["DDE_BACKEND"] = "pytorch"
import deepxde as dde
import numpy as np
import torch
import optuna
import plotly.express as px
import gc

In [47]:
kappa = 1.0
geom = dde.geometry.Interval(0, np.pi)
timedomain = dde.geometry.TimeDomain(0, 1)
geomtime = dde.geometry.GeometryXTime(geom, timedomain)

In [48]:
def pde(x, p):
    dp_t = dde.grad.jacobian(p, x, i=0, j=1)
    dp_xx = dde.grad.hessian(p, x, i=0, j=0)
    return dp_t - kappa * dp_xx

In [49]:
def exact_solution(x):
    xx = x[:, 0:1]
    tt = x[:, 1:2]
    return np.exp(-tt) * np.sin(xx)

In [50]:
points_test = geomtime.uniform_points(1000)
y_true = exact_solution(points_test)

In [51]:
fig = px.scatter(
    x=points_test[:, 0],
    y=points_test[:, 1],
    labels={"x": "Координата x", "y": "Время t"}
)

fig.add_shape(
    type="rect",
    x0=points_test[:, 0].min(), y0=points_test[:, 1].min(),
    x1=points_test[:, 0].max(), y1=points_test[:, 1].max(),
    line=dict(color="blue", width=2),
    fillcolor="rgba(0, 0, 255, 0.05)",
)

fig.update_layout(template="plotly_white")
fig.update_traces(marker=dict(size=6, color="red"))
fig.show()

In [52]:
iterations = 5000
epsilon = 1e-3

In [53]:
def output_transform(x, y):
    x_col = x[:, 0:1]
    t_col = x[:, 1:2]
    return torch.sin(x_col) + x_col * (np.pi - x_col) * t_col * y

In [54]:
def objective(trial):
  n_layers = trial.suggest_int("n_layers", 3, 6)
  n_neurons = trial.suggest_int("n_neurons", 20, 100)
  lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
  num_domain = trial.suggest_int("num_domain", 2000, 10000)

  errors = []

  for seed in range(3):
        dde.config.set_random_seed(seed)
        torch.manual_seed(seed)

        data = dde.data.TimePDE(geomtime, pde, [], num_domain=num_domain, train_distribution="uniform")

        layer_size = [2] + [n_neurons] * n_layers + [1]
        net = dde.nn.FNN(layer_size, "tanh", "Glorot normal")
        net.apply_output_transform(output_transform)

        early_stopping = dde.callbacks.EarlyStopping(min_delta=1e-6, patience=1000)

        model = dde.Model(data, net)
        model.compile("adam", lr=lr)
        model.train(iterations=iterations, callbacks=[early_stopping], display_every=5000)

        model.compile("L-BFGS")
        model.train(display_every=5000)

        y_pred = model.predict(points_test)
        l2_error = dde.metrics.l2_relative_error(y_true, y_pred)

        del model, net, data
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


        if np.isnan(l2_error) or np.isinf(l2_error) or l2_error >= epsilon:
            return float("inf")

        errors.append(float(l2_error))

  return max(errors)

def stop_callback(study, trial):
    if study.best_value < epsilon:
        study.stop()

study = optuna.create_study(direction="minimize")
study.optimize(objective, callbacks=[stop_callback])

print(study.best_params)
print(study.best_value)

[I 2026-11-15 17:47:19,844] A new study created in memory with name: no-name-7ba8785e-bd0b-405b-88c8-e4b22df263e8


Compiling model...
'compile' took 0.000240 s

Training model...

Step      Train loss    Test loss     Test metric
0         [6.64e-01]    [6.64e-01]    []  
Epoch 1001: early stopping

Best model at step 0:
  train loss: 6.64e-01
  test loss: 6.64e-01
  test metric: []

'train' took 19.767944 s

Compiling model...
'compile' took 0.001704 s

Training model...

Step      Train loss    Test loss     Test metric
1001      [3.35e-05]    [3.35e-05]    []  
1148      [2.69e-07]    [2.69e-07]    []  

Best model at step 1148:
  train loss: 2.69e-07
  test loss: 2.69e-07
  test metric: []

'train' took 4.315833 s

Compiling model...
'compile' took 0.000159 s

Training model...

Step      Train loss    Test loss     Test metric
0         [1.42e-01]    [1.42e-01]    []  
Epoch 1001: early stopping

Best model at step 0:
  train loss: 1.42e-01
  test loss: 1.42e-01
  test metric: []

'train' took 12.552446 s

Compiling model...
'compile' took 0.001661 s

Training model...

Step      Train loss   

[I 2026-11-15 17:48:32,569] Trial 0 finished with value: 6.537973968079314e-05 and parameters: {'n_layers': 5, 'n_neurons': 69, 'lr': 0.0032858059789204795, 'num_domain': 9246}. Best is trial 0 with value: 6.537973968079314e-05.


{'n_layers': 5, 'n_neurons': 69, 'lr': 0.0032858059789204795, 'num_domain': 9246}
6.537973968079314e-05


In [55]:
best_params = study.best_params

data = dde.data.TimePDE(
    geomtime, pde, [],
    num_domain=best_params['num_domain'],
    train_distribution="uniform"
)

layer_size = [2] + [best_params['n_neurons']] * best_params['n_layers'] + [1]

test_seeds = [10, 20, 30]
test_errors = []
best_model = None
min_error = float("inf")

for seed in test_seeds:
    dde.config.set_random_seed(seed)
    torch.manual_seed(seed)

    net = dde.nn.FNN(layer_size, "tanh", "Glorot normal")
    net.apply_output_transform(output_transform)

    early_stopping = dde.callbacks.EarlyStopping(min_delta=1e-6, patience=1000)

    model = dde.Model(data, net)
    model.compile("adam", lr=best_params['lr'])
    model.train(iterations=iterations, callbacks=[early_stopping], display_every=5000)

    model.compile("L-BFGS")
    model.train(display_every=5000)

    y_pred = model.predict(points_test)
    l2_error = float(dde.metrics.l2_relative_error(y_true, y_pred))
    test_errors.append(l2_error)

    if l2_error < min_error:
        min_error = l2_error
        best_model = model
    else:
        del model, net
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

all_passed = all(err < epsilon for err in test_errors)

Compiling model...
'compile' took 0.000144 s

Training model...

Step      Train loss    Test loss     Test metric
0         [1.26e+00]    [1.26e+00]    []  
Epoch 1001: early stopping

Best model at step 0:
  train loss: 1.26e+00
  test loss: 1.26e+00
  test metric: []

'train' took 17.331469 s

Compiling model...
'compile' took 0.000378 s

Training model...

Step      Train loss    Test loss     Test metric
1001      [2.52e-05]    [2.52e-05]    []  
1232      [9.59e-08]    [9.59e-08]    []  

Best model at step 1232:
  train loss: 9.59e-08
  test loss: 9.59e-08
  test metric: []

'train' took 7.377613 s

Compiling model...
'compile' took 0.000231 s

Training model...

Step      Train loss    Test loss     Test metric
0         [2.01e-01]    [2.01e-01]    []  
Epoch 1001: early stopping

Best model at step 0:
  train loss: 2.01e-01
  test loss: 2.01e-01
  test metric: []

'train' took 12.595158 s

Compiling model...
'compile' took 0.000902 s

Training model...

Step      Train loss   

In [57]:
print(l2_error < epsilon)
print(l2_error)

True
9.45765568758361e-05
